# Training Notebook — Emotion Recognition
Run this notebook on Google Colab after pushing all source files to GitHub.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_REPO = "https://github.com/YOUR_USERNAME/emotion-recognition.git"
    !git clone {GITHUB_REPO} /content/emotion-recognition
    %cd /content/emotion-recognition
    !pip install mat73 pyyaml scikit-learn -q

    from google.colab import drive
    drive.mount("/content/drive")

    import shutil, os
    os.makedirs("data/raw", exist_ok=True)
    shutil.copy("/content/drive/MyDrive/DREAMER.mat", "data/raw/DREAMER.mat")
    print("✅ Ready")
else:
    print("✅ Running locally")

In [ ]:
import torch
import sys
sys.path.insert(0, ".")

from src.utils.config   import load_config
from src.data.dataset   import DREAMERDataset
from src.models.deep_model import build_model
from src.training.trainer  import Trainer
from src.training.evaluator import evaluate_model
from torch.utils.data import DataLoader

config = load_config("configs/default.yaml")
print(config)

## Train one target (valence / arousal / dominance)

In [ ]:
TARGET = "valence"    # ← change to arousal / dominance as needed

# Build dataset
dataset = DREAMERDataset(
    mat_path    = config["data"]["raw_path"],
    target      = TARGET,
    window_sec  = config["data"]["segment_length"],
    overlap_sec = config["data"]["overlap"],
    norm_method = config["data"]["norm_method"],
    threshold   = config["labels"]["threshold"],
)
dataset.summary()

In [ ]:
# Build model
model = build_model(config["model"]["type"], config)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

In [ ]:
# Train
trainer = Trainer(model, dataset, config, target=TARGET)
history = trainer.fit()
trainer.load_best()

In [ ]:
# Evaluate on validation set
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from torch.utils.data import random_split

n_val   = int(len(dataset) * config["training"]["val_ratio"])
n_train = len(dataset) - n_val
_, val_ds = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(config["training"]["seed"])
)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

metrics = evaluate_model(
    model, val_loader, device,
    target=TARGET, history=history,
    save_dir="outputs/results"
)
print(metrics)

In [ ]:
# Save results and push to GitHub
import json, os

os.makedirs("outputs/results", exist_ok=True)
with open(f"outputs/results/metrics_{TARGET}.json", "w") as f:
    json.dump(metrics, f, indent=2)

if IN_COLAB:
    !git config --global user.email "you@example.com"
    !git config --global user.name "Your Name"
    !git add outputs/
    !git commit -m "feat: training results for {TARGET}"
    !git push
    print("✅ Pushed to GitHub")